# 8.3 端侧评估与质量保证

端侧模型的质量保证是一个系统工程，涉及精度验证、性能测试、鲁棒性评估、安全审计等多个维度。产业级部署要求每个维度都有明确的指标和通过标准。本章建立端侧模型评估的完整框架。

## 为什么端侧评估不同于云端？

| 维度 | 云端评估 | 端侧评估 | 差异原因 |
|------|---------|---------|----------|
| **量化影响** | FP16/FP32, 精度损失小 | INT4/INT8, 精度损失显著 | 端侧必须量化 |
| **硬件差异** | 统一GPU | 碎片化NPU/CPU | 不同硬件数值行为不同 |
| **资源约束** | 充足 | 有限(内存/算力/功耗) | 约束导致功能裁剪 |
| **用户场景** | 批处理为主 | 交互式为主 | 延迟要求更严格 |
| **安全风险** | 服务端可控 | 模型在用户设备上 | 模型可被提取/篡改 |

## 8.3.1 精度验证体系

### 四级精度验证

```
第一级: 逐层余弦相似度 → 确保每层输出与原始模型一致
第二级: 端到端PPL对比 → 确保语言建模能力未退化
第三级: 基准测试集评估 → 确保任务能力未退化
第四级: 人工评估 → 确保用户体验可接受
```

### 各级验证的通过标准

| 级别 | 指标 | 通过标准 | 失败处理 |
|------|------|---------|----------|
| **L1 逐层** | 余弦相似度 | >0.999 | 定位敏感层，调整量化策略 |
| **L2 PPL** | PPL增加 | <0.5 (INT8), <1.0 (INT4) | 检查量化配置，尝试GPTQ/AWQ |
| **L3 基准** | 准确率下降 | <2% (通用), <5% (量化) | 针对退化任务微调/QAT |
| **L4 人工** | 人类偏好 | 胜率>45% (vs 原始) | 分析bad case，迭代优化 |

### 量化精度损失的来源与量化

| 损失来源 | 影响程度 | 检测方法 | 缓解策略 |
|---------|---------|---------|----------|
| **权重量化** | 高(精度敏感层) | L1逐层对比 | 混合精度(敏感层FP16) |
| **KV Cache量化** | 中(长上下文) | 长文本PPL | KV FP8或按层混合精度 |
| **激活值量化** | 高(outlier) | 激活值范围分析 | SmoothQuant/GPTQ |
| **计算精度** | 低(累加误差) | 多步推理对比 | 适当提高累加精度 |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import numpy as np

@dataclass
class LayerValidationResult:
    layer_name: str
    cosine_similarity: float
    max_abs_diff: float
    mean_abs_diff: float
    passed: bool

class AccuracyValidator:
    """端侧模型精度验证器"""
    def __init__(self, baseline_ppl, quant_config):
        self.baseline_ppl = baseline_ppl
        self.quant_config = quant_config
        self.layer_results: List[LayerValidationResult] = []
        self.e2e_results: Dict = {}

    def validate_layer(self, layer_name, baseline_output, quant_output, threshold=0.999):
        baseline_flat = baseline_output.flatten()
        quant_flat = quant_output.flatten()
        cos_sim = np.dot(baseline_flat, quant_flat) / (
            np.linalg.norm(baseline_flat) * np.linalg.norm(quant_flat) + 1e-10)
        max_diff = np.max(np.abs(baseline_flat - quant_flat))
        mean_diff = np.mean(np.abs(baseline_flat - quant_flat))
        result = LayerValidationResult(
            layer_name=layer_name, cosine_similarity=cos_sim,
            max_abs_diff=max_diff, mean_abs_diff=mean_diff,
            passed=cos_sim >= threshold
        )
        self.layer_results.append(result)
        return result

    def validate_ppl(self, quant_ppl):
        ppl_increase = quant_ppl - self.baseline_ppl
        max_allowed = {'int4': 1.0, 'int8': 0.5, 'fp16': 0.1}.get(self.quant_config, 0.5)
        return {'ppl_increase': ppl_increase, 'max_allowed': max_allowed,
                'passed': ppl_increase <= max_allowed}

    def validate_benchmark(self, baseline_scores, quant_scores, max_drop_pct=5):
        results = {}
        all_passed = True
        for task, baseline in baseline_scores.items():
            quant = quant_scores.get(task, 0)
            drop_pct = (baseline - quant) / baseline * 100 if baseline > 0 else 0
            passed = drop_pct <= max_drop_pct
            results[task] = {'baseline': baseline, 'quant': quant,
                             'drop_pct': drop_pct, 'passed': passed}
            if not passed:
                all_passed = False
        return {'tasks': results, 'all_passed': all_passed}

    def summary(self):
        failed_layers = [r for r in self.layer_results if not r.passed]
        return {
            'total_layers': len(self.layer_results),
            'failed_layers': len(failed_layers),
            'failed_layer_names': [r.layer_name for r in failed_layers],
            'min_cosine_sim': min(r.cosine_similarity for r in self.layer_results) if self.layer_results else 0,
        }

validator = AccuracyValidator(baseline_ppl=5.12, quant_config='int4')

np.random.seed(42)
layer_names = ['embed', 'layer.0.attn', 'layer.0.ffn', 'layer.1.attn', 'layer.1.ffn',
               'layer.15.attn', 'layer.15.ffn', 'layer.31.attn', 'layer.31.ffn', 'lm_head']
for name in layer_names:
    baseline = np.random.randn(1, 128).astype(np.float32)
    noise_level = 0.001 if 'attn' in name else 0.005 if 'ffn' in name else 0.01
    quant = baseline + np.random.randn(*baseline.shape) * noise_level
    validator.validate_layer(name, baseline, quant)

print("=== L1 逐层精度验证 ===")
print(f"{'层名':<20} {'余弦相似度':<12} {'最大绝对差':<12} {'是否通过'}")
print("-" * 60)
for r in validator.layer_results:
    status = '✓' if r.passed else '✗'
    print(f"{r.layer_name:<20} {r.cosine_similarity:<12.6f} {r.max_abs_diff:<12.6f} {status}")

summary = validator.summary()
print(f"\n总计: {summary['total_layers']}层, 失败: {summary['failed_layers']}层")
if summary['failed_layers'] > 0:
    print(f"失败层: {summary['failed_layer_names']}")
    print(f"建议: 对失败层使用FP16精度(混合精度量化)")

print(f"\n=== L2 PPL验证 ===")
ppl_result = validator.validate_ppl(quant_ppl=5.85)
status = '✓ 通过' if ppl_result['passed'] else '✗ 不通过'
print(f"PPL增加: {ppl_result['ppl_increase']:.2f} (阈值<{ppl_result['max_allowed']}) → {status}")

print(f"\n=== L3 基准测试验证 ===")
baseline_scores = {'MMLU': 62.5, 'HumanEval': 38.0, 'GSM8K': 52.0, 'C-Eval': 58.0}
quant_scores = {'MMLU': 60.8, 'HumanEval': 36.5, 'GSM8K': 49.0, 'C-Eval': 56.2}
bench_result = validator.validate_benchmark(baseline_scores, quant_scores, max_drop_pct=5)
for task, r in bench_result['tasks'].items():
    status = '✓' if r['passed'] else '✗'
    print(f"  {task}: {r['baseline']:.1f}→{r['quant']:.1f} (下降{r['drop_pct']:.1f}%) {status}")
print(f"总体: {'✓ 全部通过' if bench_result['all_passed'] else '✗ 存在不通过项'}")

## 8.3.2 性能测试与SLA

### 端侧推理SLA定义

| SLA等级 | TTFT | ITL | 吞吐 | 可用性 | 适用场景 |
|---------|------|-----|------|--------|----------|
| **S级** | <300ms | <30ms | >30 tok/s | 99.9% | 实时对话(旗舰设备) |
| **A级** | <500ms | <50ms | >15 tok/s | 99.5% | 助手类应用 |
| **B级** | <1000ms | <100ms | >10 tok/s | 99% | 后台处理 |
| **C级** | <3000ms | <200ms | >5 tok/s | 95% | 低端设备/大模型 |

### 性能测试方法论

| 测试类型 | 目的 | 方法 | 通过标准 |
|---------|------|------|----------|
| **冷启动测试** | 首次加载性能 | 杀进程后首次推理 | TTFT<3s |
| **稳态测试** | 正常推理性能 | 100次推理取P50/P90/P99 | 满足SLA |
| **压力测试** | 长时间稳定性 | 30分钟连续推理 | 性能下降<30% |
| **边界测试** | 极端输入性能 | 最大seq_len/最大batch | 不崩溃 |
| **干扰测试** | 后台应用影响 | 同时运行其他应用 | 性能下降<20% |

## 8.3.3 鲁棒性与安全评估

### 鲁棒性评估维度

| 维度 | 测试内容 | 评估方法 | 通过标准 |
|------|---------|---------|----------|
| **输入鲁棒性** | 特殊字符/超长输入/空输入 | 不崩溃+合理输出 | 0%崩溃率 |
| **对抗鲁棒性** | 对抗样本/越狱攻击 | 拒绝率+安全过滤 | 拒绝率>95% |
| **多语言鲁棒性** | 中英日韩混合输入 | 输出质量不退化 | PPL增加<1.0 |
| **数值鲁棒性** | 极大/极小数值输入 | 不产生NaN/Inf | 0%数值异常 |
| **多轮对话** | 长对话上下文累积 | 不重复/不退化 | 重复率<10% |

### 安全评估清单

- [ ] 越狱攻击防御测试（常见攻击模板+自动化测试）
- [ ] 有害内容过滤测试（暴力/色情/歧视等类别）
- [ ] 隐私信息泄露测试（PII检测+脱敏验证）
- [ ] 模型提取攻击评估（查询次数与模型还原度）
- [ ] 数据投毒防御验证（训练数据完整性检查）

In [ ]:
class EndToEndEvaluator:
    """端到端评估器"""
    def __init__(self, model_name, quant_config, target_device):
        self.model_name = model_name
        self.quant_config = quant_config
        self.target_device = target_device
        self.results = {}

    def full_evaluation(self):
        self.results = {
            'accuracy': self._eval_accuracy(),
            'performance': self._eval_performance(),
            'robustness': self._eval_robustness(),
            'security': self._eval_security(),
        }
        self.results['overall'] = all(
            r['passed'] for r in self.results.values()
        )
        return self.results

    def _eval_accuracy(self):
        ppl_ok = True
        bench_ok = True
        return {'ppl_pass': ppl_ok, 'benchmark_pass': bench_ok, 'passed': ppl_ok and bench_ok}

    def _eval_performance(self):
        sla_ok = True
        stress_ok = True
        return {'sla_pass': sla_ok, 'stress_pass': stress_ok, 'passed': sla_ok and stress_ok}

    def _eval_robustness(self):
        input_ok = True
        adversarial_ok = True
        return {'input_pass': input_ok, 'adversarial_pass': adversarial_ok,
                'passed': input_ok and adversarial_ok}

    def _eval_security(self):
        jailbreak_ok = True
        content_filter_ok = True
        return {'jailbreak_pass': jailbreak_ok, 'content_filter_pass': content_filter_ok,
                'passed': jailbreak_ok and content_filter_ok}

    def report(self):
        if not self.results:
            self.full_evaluation()
        print(f"=== 端到端评估报告 ===")
        print(f"模型: {self.model_name} ({self.quant_config})")
        print(f"目标设备: {self.target_device}")
        print(f"\n{'维度':<15} {'状态':<8} {'详情'}")
        print("-" * 50)
        for dim, result in self.results.items():
            if dim == 'overall':
                continue
            status = '✓ 通过' if result['passed'] else '✗ 不通过'
            details = ', '.join(f"{k}:{'✓' if v else '✗'}" for k, v in result.items() if k != 'passed')
            print(f"{dim:<15} {status:<8} {details}")
        overall = '✓ 全部通过' if self.results.get('overall') else '✗ 存在不通过项'
        print(f"\n总体: {overall}")

evaluator = EndToEndEvaluator('Qwen2.5-7B-Instruct', 'W4A16', 'Snapdragon 8 Gen3')
evaluator.report()

print(f"\n=== 端侧评估总结 ===")
print(f"1. 四级精度验证: 逐层→PPL→基准→人工，逐层深入")
print(f"2. SLA分级: S/A/B/C级，不同场景不同标准")
print(f"3. 五类性能测试: 冷启动+稳态+压力+边界+干扰")
print(f"4. 鲁棒性评估: 输入/对抗/多语言/数值/多轮对话")
print(f"5. 安全评估: 越狱/有害内容/隐私/模型提取/数据投毒")
print(f"6. 评估自动化: CI/CD集成，每次模型更新自动跑全量评估")